# Chapter 29 – Worked examples

This notebook runs the examples of [Chapter 29](../index.md) live: the PTDF as the automatic derivative of a DC
power flow (`jax_ptdf.py`), its extension to the susceptances (`jax_susceptance.py`), value iteration on a tiny MDP
(`value_iteration.py`), the size of the topology action space (`action_space.py`) and the five-bus
topology-control environment with its agents (`grid_env.py`). The cells only call the modules.

Run it from this folder (the kernel's working directory must be `src/`).

In [ ]:
import action_space
import grid_env
import jax_ptdf
import jax_susceptance
import numpy as np
import value_iteration

## The PTDF by automatic differentiation

`jax_ptdf.flows` solves the DC power flow of a three-bus ring in JAX; `jax.jacfwd` of it is the PTDF of
Chapter 17, and the Jacobian with respect to the susceptances (Exercise 3) shows how a line's stiffness moves
flow around the loop.

In [ ]:
print('flows :', np.round(np.asarray(jax_ptdf.flows(jax_susceptance.P)), 4))
print('PTDF  :\n', np.round(np.asarray(jax_ptdf.ptdf()), 4))
print('d flows / d b:\n', np.round(np.asarray(jax_susceptance.dflows_db()), 4))

## Value iteration on a two-state MDP

The "secure / overloaded" MDP of Exercise 1 with $\gamma = 0.9$: the Bellman optimality operator is a
contraction, so the iterates converge to $V^*$; the greedy policy waits when secure and switches when overloaded.

In [ ]:
v, policy, history = value_iteration.value_iteration()
for k in (1, 2, 3):
    print(f'V_{k} = {np.round(history[k], 4)}')
print(f'V* = {np.round(v, 4)} after {len(history) - 1} iterations, policy {policy}')

## How big is the action space?

Six elements on two busbars, fourteen substations: the joint space is astronomically large, which is why agents
use unitary actions, per-substation action sets and simulation-based pruning.

In [ ]:
d = 6
print('labelled assignments', action_space.assignments(d), end=' | ')
print('up to symmetry', action_space.assignments_up_to_symmetry(d))
print('ToOp splits', action_space.toop_splits(d), '| Grid2Op unitary topologies', action_space.unitary_topologies(d))
print(f'14 substations jointly: {action_space.joint(action_space.toop_splits(d) + 1):.2e}')

## A learned agent against exhaustive one-step search

`grid_env.GridEnv` is a five-bus stand-in for a Grid2Op environment: one day of 288 steps, a peak that
overloads the weak tie line 2–3 in the meshed topology, and actions that open one line. The greedy agent simulates
every action when a line passes 90 %; tabular Q-learning over the actions the greedy agent uses most learns the
same behaviour from reward alone.

In [ ]:
peak = grid_env.load_profile(np.arange(grid_env.N_STEPS)).max()
print('loading at the peak, meshed:', np.round(grid_env.rho(0, peak), 2))
actions = grid_env.frequent_actions()
q = grid_env.q_learning(actions)
results = grid_env.compare(range(10), q=q, actions=actions)
for name, runs in results.items():
    survival = [r['survival'] for r in runs]
    print(f'{name:18s} survival median {np.median(survival):.0f}  reward {np.mean([r["reward"] for r in runs]):6.2f}')